 Captioning
Caption a dataset on Drive with Qwen3-VL (abliterated options), JoyCaption or WD tags.

1. **Runtime > Change runtime type > T4 GPU**
2. **Secrets** (key icon, left), allow notebook access: `HF_TOKEN` (gated or private repos),
   `CIVITAI_TOKEN` (Civitai downloads), `TS_AUTHKEY` (Tailscale, tick Reusable + Ephemeral; only the app notebooks).
3. Run **Setup** (re-runs skip finished steps), then the cells below in order.

In [ ]:
#@title Setup: repo, venv, secrets, Drive
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/kopeng-mi/SDXL_Research.git"  #@param {type:"string"}
BRANCH = "step5-notebooks"  #@param {type:"string"}
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/SDXLR"  #@param {type:"string"}
GROUPS = 'caption'  # venv install groups for this notebook (None = no shared venv)
REPO = Path("/content/SDXL_Research")

if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "--recurse-submodules", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
else:
    # follow BRANCH even if an earlier run cloned another one
    subprocess.run(["git", "fetch", "-q", "origin", BRANCH], cwd=REPO, check=True)
    subprocess.run(["git", "checkout", "-q", "-B", BRANCH, f"origin/{BRANCH}"], cwd=REPO, check=True)
    subprocess.run(["git", "submodule", "update", "-q", "--init", "--recursive"], cwd=REPO, check=True)

# The kernel only orchestrates (stdlib); GPU work runs in venv subprocesses,
# so this kernel never holds VRAM.
sys.path.insert(0, str(REPO / "src"))
from sdxlr import nb
if GROUPS is not None:
    r = subprocess.run([sys.executable, "-m", "sdxlr.bootstrap", "--groups", GROUPS], stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT, text=True, env={**os.environ, "PYTHONPATH": str(REPO / "src")})
    print(r.stdout[-3000:])
    assert r.returncode == 0, "setup failed (see above)"
found = nb.load_secrets()
print("secrets:", ", ".join(f"{k} {'found' if v else 'missing'}" for k, v in found.items()))
ROOT = nb.drive_mount(DRIVE_ROOT) if USE_DRIVE else None
print("drive:", ROOT or "off")

# 1. Settings
The dataset lives on Drive in `SDXLR/datasets/<DS_NAME>`; captions are written there as they finish.

In [ ]:
#@title Settings
DS_NAME = "my_dataset"  #@param {type:"string"}
#@markdown **SRC**: where the photos come from (comma list): a Drive folder, a .zip link, an HF file
#@markdown `org/repo/file.zip`, or `test` (5 public photos). Leave empty to use the Drive dataset as it is.
SRC = ""  #@param {type:"string"}
MODEL = "qwen3vl-4b"  #@param ["qwen3vl-4b", "qwen3vl-2b", "qwen3vl-4b-abl", "qwen3vl-8b-abl", "qwen35-4b-abl (exp)", "qwen35-9b-abl (exp)", "joy", "wd"] {allow-input: true}
#@markdown **STYLE**: `id` for a person/character/object LoRA (no face or body words), `detailed` for style or
#@markdown concept LoRAs, `tags` for booru tags (WD tagger, ignores MODEL).
STYLE = "id"  #@param ["id", "detailed", "tags"]
TARGET = "sdxl"  #@param ["sdxl", "zimage"]
#@markdown TARGET sets the length: sdxl 30 to 50 words, zimage 80 to 150.
TRIGGER = "ohwx"  #@param {type:"string"}
CLASS_WORD = "woman"  #@param {type:"string"}
MAX_WORDS = 0  #@param {type:"integer"}
CUSTOM_PROMPT = ""  #@param {type:"string"}
#@markdown CUSTOM_PROMPT replaces the style prompt; `{cls}` and `{words}` are filled in.
QUANT = "auto"  #@param ["auto", "fp16", "nf4"]
BATCH = 0  #@param {type:"integer"}
OVERWRITE = False  #@param {type:"boolean"}

from sdxlr import drive, plan, paths
LOCAL = paths.DATASETS / DS_NAME
DRIVE_DS = (ROOT or drive.ROOT) / "datasets" / DS_NAME
repo, params_b, loader = paths.CAPTIONERS.get(MODEL, (MODEL, 4.0, "qwen"))
q = plan.caption_quant(params_b) if QUANT == "auto" else QUANT
print(f"dataset: {LOCAL}  (Drive: {DRIVE_DS if ROOT else 'off'})")
print(f"captioner: {repo}, {q}, about {plan.caption_vram(params_b, q):.1f} GB of the T4's 15 GB")
cap_args = ["-m", "sdxlr.caption", "--model", MODEL, "--style", STYLE, "--target", TARGET, "--trigger", TRIGGER,
            "--class-word", CLASS_WORD, "--max-words", MAX_WORDS, "--quant", QUANT, "--batch", BATCH]
if CUSTOM_PROMPT:
    cap_args += ["--prompt", CUSTOM_PROMPT]

# 2. Import
Copies the Drive dataset to the VM (fast reads), adds SRC (duplicates skipped, phone photos turned upright),
copies the result back to Drive and prints the checks.

In [ ]:
#@title Import
CHECK_RES = 1024  #@param {type:"integer"}
#@markdown CHECK_RES: training size to check against (SDXL 1024, Z-Image 512 or 768).
LOCAL.mkdir(parents=True, exist_ok=True)
if ROOT and DRIVE_DS.exists():
    print(f"from Drive: {drive.pull(DRIVE_DS, LOCAL)} files copied")
srcs = [s.strip() for s in SRC.split(",") if s.strip()]
nb.run(["-m", "sdxlr.data", "--dest", LOCAL, *srcs, "--res", CHECK_RES, "--kind", TARGET])
if ROOT:
    print(f"to Drive: {drive.pull(LOCAL, DRIVE_DS)} files copied")

# 3. Preview
Captions the first few photos into a scratch folder, so you can change the settings and run it again.

In [ ]:
#@title Preview
N = 4  #@param {type:"integer"}
import shutil
PREVIEW = paths.WORK / "caption_preview"
shutil.rmtree(PREVIEW, ignore_errors=True)
nb.run(cap_args + ["--images", LOCAL, "--out", PREVIEW, "--limit", N, "--overwrite"])
nb.show_images(sorted(p for p in LOCAL.iterdir() if p.suffix.lower() in (".png", ".jpg", ".jpeg", ".webp"))[:N],
               captions=PREVIEW)

# 4. Caption all
Photos that already have a caption are skipped unless OVERWRITE is on. Each caption is copied to Drive
as soon as it is written; stop and run again to continue.

In [ ]:
#@title Caption all
args = cap_args + ["--images", LOCAL]
if OVERWRITE:
    args += ["--overwrite"]
if ROOT:
    args += ["--mirror", DRIVE_DS]
nb.run(args)

# 5. Review
Flags missing captions, captions over the word budget, and captions not starting with the trigger.
FIND/REPLACE edits every caption (and its Drive copy). Run the last cell before closing the tab.

In [ ]:
#@title Review
FIND = ""  #@param {type:"string"}
REPLACE = ""  #@param {type:"string"}
SHOW = 8  #@param {type:"integer"}
MAKE_ZIP = False  #@param {type:"boolean"}
from sdxlr import review
if FIND:
    print(f"replaced in {review.replace(LOCAL, FIND, REPLACE, DRIVE_DS if ROOT else None)} captions")
print(review.report(LOCAL, TARGET, TRIGGER))
nb.show_images(sorted(p for p in LOCAL.iterdir() if p.suffix.lower() in (".png", ".jpg", ".jpeg", ".webp"))[:SHOW])
if MAKE_ZIP:
    z = review.zip_dataset(LOCAL, paths.WORK / f"{DS_NAME}.zip")
    from google.colab import files
    files.download(str(z))

# Finish
Drive writes back lazily: run this before closing the tab, then **Runtime > Disconnect and delete runtime**.

In [ ]:
#@title Save to Drive and finish
if ROOT:
    drive.flush_and_unmount()